# First Queries

DuckDB is an analytical SQL database that runs inside Julia. This tutorial creates a table, asks it questions in SQL and brings the answers into Julia as columns.

It needs DuckDB.jl, DBInterface.jl and Tables.jl, which Pollis installs with its built-in database. Everything happens in an **in-memory** database, so nothing is saved and the built-in Pollis database is left untouched: a DuckDB file can be opened for writing by only one process at a time, and the Julia REPL may already hold it.

In [ ]:
# DuckDB is built into Pollis: use its environment if this project does not have DuckDB
pollis_env = joinpath(homedir(), ".pollis", "julia", "environments", "v$(VERSION.major).$(VERSION.minor)")
isdir(pollis_env) && !(pollis_env in LOAD_PATH) && push!(LOAD_PATH, pollis_env)

using DuckDB, DBInterface, Tables, Statistics

con = DBInterface.connect(DuckDB.DB, ":memory:")

## Create a Table

`DBInterface.execute` sends one SQL statement to the database. This one builds a table of 1000 sales from DuckDB's `range` function, so the tutorial needs no files.

In [ ]:
DBInterface.execute(con, """
CREATE TABLE sales AS
SELECT i AS id,
       ['north', 'south', 'east', 'west'][1 + i % 4] AS region,
       DATE '2026-01-01' + (i % 365)::INTEGER AS day,
       round(10 + (i * 37 % 900) / 10, 2) AS price,
       1 + i % 7 AS qty
FROM range(1000) t(i)
""")
Tables.columntable(DBInterface.execute(con, "DESCRIBE sales"))

## Filter and Sort

A query returns a cursor; `Tables.columntable` turns it into a named tuple of typed Julia vectors. `WHERE` keeps matching rows, `ORDER BY` sorts and `LIMIT` keeps the first ones.

In [ ]:
top = Tables.columntable(DBInterface.execute(con, """
SELECT id, region, day, price, qty
FROM sales
WHERE region = 'north' AND qty >= 5
ORDER BY price DESC
LIMIT 5
"""))

In [ ]:
top.price, eltype(top.day)

## Aggregate

`GROUP BY` collapses rows into groups and aggregates each one. Only the four result rows travel to Julia, however large the table.

In [ ]:
by_region = Tables.columntable(DBInterface.execute(con, """
SELECT region,
       count(*) AS orders,
       round(avg(price), 2) AS mean_price,
       round(sum(price * qty), 2) AS revenue
FROM sales
GROUP BY region
ORDER BY revenue DESC
"""))

## Parameters

Values that change belong in parameters, not pasted into the SQL: `?` placeholders are filled from a tuple. The statement is prepared once and run for each value.

In [ ]:
stmt = DBInterface.prepare(con, "SELECT count(*) AS n FROM sales WHERE price > ?")
for p in (25, 50, 75)
	n = first(Tables.columntable(DBInterface.execute(stmt, (p,))).n)
	println("price > ", p, ": ", n, " sales")
end

## Check in Julia

The same numbers can be computed in Julia from the full table. SQL and Julia should agree.

In [ ]:
all_sales = Tables.columntable(DBInterface.execute(con, "SELECT * FROM sales"))
north = all_sales.region .== "north"
julia_mean = round(mean(all_sales.price[north]), digits=2)
sql_mean = by_region.mean_price[findfirst(==("north"), by_region.region)]
julia_mean, sql_mean, julia_mean == sql_mean

## NULL and missing

SQL's `NULL` arrives in Julia as `missing`. Aggregates skip it, like `skipmissing`: `count(price)` counts the prices that exist, `count(*)` counts rows.

In [ ]:
DBInterface.execute(con, "INSERT INTO sales VALUES (1000, 'north', DATE '2026-12-31', NULL, 1)")
Tables.columntable(DBInterface.execute(con, "SELECT count(*) AS rows, count(price) AS prices, round(avg(price), 2) AS mean_price FROM sales"))

In [ ]:
prices = Tables.columntable(DBInterface.execute(con, "SELECT price FROM sales")).price
eltype(prices), count(ismissing, prices), round(mean(skipmissing(prices)), digits=2)

## Close

Closing the connection frees the in-memory database.

In [ ]:
DBInterface.close!(con)